# E-Commerce Sales & Customer Analytics

## 04 — Business Analysis

### 1. Business Objective

The objective of this analysis is to transform the findings from
exploratory data analysis into meaningful business insights and
recommendations.

The analysis focuses on:

- Sales performance
- Product performance
- Customer behavior
- Delivery operations
- Customer satisfaction

The goal is to identify key opportunities and potential challenges
that can support data-driven decision-making.

### 2. Import Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

### 3. Load Processed Data

In [2]:
data_path = "../data/processed/"

customers = pd.read_csv(data_path + "customers_clean.csv")
orders = pd.read_csv(data_path + "orders_clean.csv")
order_items = pd.read_csv(data_path + "order_items_clean.csv")
products = pd.read_csv(data_path + "products_clean.csv")
payments = pd.read_csv(data_path + "payments_clean.csv")
reviews = pd.read_csv(data_path + "reviews_clean.csv")

In [3]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors='coerce'
    )

### 4. Create Analysis Dataset

In [4]:
analysis_df = (
    order_items
    .merge(
        orders[
            [
                'order_id',
                'customer_id',
                'order_status',
                'order_purchase_timestamp',
                'delivery_days',
                'delivery_delay_days'
            ]
        ],
        on='order_id',
        how='left'
    )
    .merge(
        products[
            [
                'product_id',
                'product_category_name'
            ]
        ],
        on='product_id',
        how='left'
    )
)

In [5]:
analysis_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,delivery_days,delivery_delay_days,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,7.0,-9.0,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,16.0,-3.0,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,7.0,-14.0,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,6.0,-6.0,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,25.0,-16.0,ferramentas_jardim


In [6]:
analysis_df.shape

(112650, 13)

### 5. Sales Performance Analysis
###    5.1 Key Performance Indicators

In [7]:
total_revenue = order_items['price'].sum()

total_orders = orders['order_id'].nunique()

delivered_orders = orders[
    orders['order_status'] == 'delivered'
].copy()

delivered_orders_count = (
    delivered_orders['order_id'].nunique()
)

delivered_revenue = (
    analysis_df[
        analysis_df['order_status'] == 'delivered'
    ]['price'].sum()
)

delivered_aov = (
    delivered_revenue / delivered_orders_count
)

In [8]:
sales_kpis = pd.DataFrame({
    'Metric': [
        'Total Orders',
        'Delivered Orders',
        'Total Revenue',
        'Delivered Revenue',
        'Delivered AOV'
    ],
    'Value': [
        total_orders,
        delivered_orders_count,
        total_revenue,
        delivered_revenue,
        delivered_aov
    ]
})

sales_kpis

,Metric,Value
0,Total Orders,9.944100e+04
1,Delivered Orders,9.647800e+04
2,Total Revenue,1.359164e+07
3,Delivered Revenue,1.322150e+07
4,Delivered AOV,1.370416e+02


### 5.2 Monthly Sales Trends

In [34]:
analysis_df['order_month'] = (
    analysis_df['order_purchase_timestamp']
    .dt.to_period('M')
    .astype(str)
)

monthly_sales = (
    analysis_df
    .groupby('order_month')
    .agg(
        revenue=('price', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
)

monthly_sales.head()

,order_month,revenue,orders
0,2016-09,267.36,3
1,2016-10,49507.66,308
2,2016-12,10.90,1
3,2017-01,120312.87,789
4,2017-02,247303.02,1733


### 5.3 Revenue Growth

In [35]:
monthly_sales['revenue_growth_pct'] = (
    monthly_sales['revenue']
    .pct_change() * 100
)

monthly_sales.head(10)

,order_month,revenue,orders,revenue_growth_pct
0,2016-09,267.36,3,NaN
1,2016-10,49507.66,308,1.841723e+04
2,2016-12,10.90,1,-9.997798e+01
3,2017-01,120312.87,789,1.103688e+06
4,2017-02,247303.02,1733,1.055499e+02
5,2017-03,374344.30,2641,5.137069e+01
6,2017-04,359927.23,2391,-3.851286e+00
7,2017-05,506071.14,3660,4.060374e+01
8,2017-06,433038.60,3217,-1.443128e+01
9,2017-07,498031.48,3969,1.500857e+01


In [36]:
monthly_sales.sort_values(
    'revenue_growth_pct',
    ascending=False
).head(10)

,order_month,revenue,orders,revenue_growth_pct
3,2017-01,120312.87,789,1.103688e+06
1,2016-10,49507.66,308,1.841723e+04
4,2017-02,247303.02,1733,1.055499e+02
13,2017-11,1010271.37,7451,5.209904e+01
5,2017-03,374344.30,2641,5.137069e+01
7,2017-05,506071.14,3660,4.060374e+01
15,2018-01,950030.36,7220,2.770699e+01
17,2018-03,983213.44,7188,1.646982e+01
10,2017-08,573971.68,4293,1.524807e+01
9,2017-07,498031.48,3969,1.500857e+01


### 6. Product Performance Analysis

In [33]:
category_performance = (
    analysis_df
    .groupby('product_category_name')
    .agg(
        revenue=('price', 'sum'),
        orders=('order_id', 'nunique'),
        items_sold=('order_id', 'count')
    )
    .sort_values(
        'revenue',
        ascending=False
    )
)

category_performance.head(10)

,revenue,orders,items_sold
product_category_name,,,
beleza_saude,1258681.34,8836,9670
relogios_presentes,1205005.68,5624,5991
cama_mesa_banho,1036988.68,9417,11115
esporte_lazer,988048.97,7720,8641
informatica_acessorios,911954.32,6689,7827
moveis_decoracao,729762.49,6449,8334
cool_stuff,635290.85,3632,3796
utilidades_domesticas,632248.66,5884,6964
automotivo,592720.11,3897,4235


### 6.1 High-Revenue Categories

In [25]:
high_revenue_categories = (
    category_performance
    .sort_values('revenue', ascending=False)
    .head(10)
)

high_revenue_categories

,revenue,orders,items_sold
product_category_name,,,
beleza_saude,1258681.34,8836,9670
relogios_presentes,1205005.68,5624,5991
cama_mesa_banho,1036988.68,9417,11115
esporte_lazer,988048.97,7720,8641
informatica_acessorios,911954.32,6689,7827
moveis_decoracao,729762.49,6449,8334
cool_stuff,635290.85,3632,3796
utilidades_domesticas,632248.66,5884,6964
automotivo,592720.11,3897,4235


###  6.2 High-Volume Categories

In [26]:
high_volume_categories = (
    category_performance
    .sort_values('orders', ascending=False)
    .head(10)
)

high_volume_categories

,revenue,orders,items_sold
product_category_name,,,
cama_mesa_banho,1036988.68,9417,11115
beleza_saude,1258681.34,8836,9670
esporte_lazer,988048.97,7720,8641
informatica_acessorios,911954.32,6689,7827
moveis_decoracao,729762.49,6449,8334
utilidades_domesticas,632248.66,5884,6964
relogios_presentes,1205005.68,5624,5991
telefonia,323667.53,4199,4545
automotivo,592720.11,3897,4235


### 6.3 Revenue vs Order Volume

In [27]:
category_performance['revenue_per_order'] = (
    category_performance['revenue']
    / category_performance['orders']
)

category_performance.sort_values(
    'revenue_per_order',
    ascending=False
).head(10)

,revenue,orders,items_sold,revenue_per_order
product_category_name,,,,
pcs,222963.13,181,203,1231.840497
portateis_casa_forno_e_cafe,47445.71,75,76,632.609467
eletrodomesticos_2,113317.74,234,238,484.263846
agro_industria_e_comercio,72530.47,182,212,398.519066
instrumentos_musicais,191498.88,628,680,304.934522
eletroportateis,190648.58,630,679,302.616794
portateis_cozinha_e_preparadores_de_alimentos,3968.53,14,15,283.466429
telefonia_fixa,59583.00,217,264,274.576037
construcao_ferramentas_seguranca,40544.52,167,194,242.781557


### 7. Customer Analysis
###    7.1 Customer Geographic Concentration

In [15]:
customer_geography = (
    customers['customer_state']
    .value_counts()
    .reset_index()
)

customer_geography.columns = [
    'State',
    'Customers'
]

customer_geography.head(10)

,State,Customers
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


###  7.2 Repeat Customer Behavior

In [16]:
customer_orders = (
    customers['customer_unique_id']
    .value_counts()
)

repeat_customer_count = (
    customer_orders > 1
).sum()

repeat_customer_rate = (
    repeat_customer_count
    / customers['customer_unique_id'].nunique()
) * 100

repeat_customer_rate

np.float64(3.1187562437562435)

### 8. Operational Performance Analysis
###    8.1 Delivery Performance KPIs

In [17]:
delivered_orders['is_late'] = (
    delivered_orders['delivery_delay_days'] > 0
)

### 8.2 Late Delivery Analysis

In [28]:
late_delivery_summary = (
    delivered_orders['is_late']
    .value_counts()
    .rename(index={
        False: 'On-Time or Early',
        True: 'Late'
    })
)

late_delivery_summary

is_late
On-Time or Early    89944
Late                 6534
Name: count, dtype: int64

In [43]:
average_delivery_days = round(
    delivered_orders['delivery_days'].mean(),
    2
)

late_delivery_rate = round(
    delivered_orders['is_late'].mean() * 100,
    2
)

In [44]:
delivery_kpis = pd.DataFrame({
    'Metric': [
        'Average Delivery Days',
        'Late Delivery Rate'
    ],
    'Value': [
        average_delivery_days,
        late_delivery_rate
    ]
})

delivery_kpis

,Metric,Value
0,Average Delivery Days,12.09
1,Late Delivery Rate,6.77


###   8.3 Categories with Delivery Challenges

In [39]:
delivery_category = (
    analysis_df
    .merge(
        delivered_orders[
            [
                'order_id',
                'is_late'
            ]
        ],
        on='order_id',
        how='inner'
    )
)

In [40]:
category_delivery_performance = (
    delivery_category
    .groupby('product_category_name')
    .agg(
        average_delivery_days=(
            'delivery_days',
            'mean'
        ),
        late_delivery_rate=(
            'is_late',
            'mean'
        )
    )
)

category_delivery_performance[
    'late_delivery_rate'
] = (
    category_delivery_performance[
        'late_delivery_rate'
    ] * 100
)

category_delivery_performance.sort_values(
    'late_delivery_rate',
    ascending=False
).head(10)

,average_delivery_days,late_delivery_rate
product_category_name,,
moveis_colchao_e_estofado,13.891892,13.513514
casa_conforto_2,14.066667,13.333333
audio,12.883978,11.602210
artigos_de_natal,15.300000,10.000000
fashion_underwear_e_moda_praia,13.275591,9.448819
casa_conforto,13.039627,9.324009
livros_tecnicos,10.231939,7.984791
moveis_escritorio,20.386691,7.973621
bebes,12.056338,7.679410


### 9. Customer Satisfaction Analysis
###    9.1 Review Score KPIs

In [29]:
average_review_score = (
    reviews['review_score'].mean()
)

average_review_score

np.float64(4.08642062404257)

In [30]:
five_star_rate = (
    (reviews['review_score'] == 5).mean() * 100
)

five_star_rate

np.float64(57.776344432798524)

In [32]:
review_kpis = pd.DataFrame({
    'Metric': [
        'Average Review Score',
        '5-Star Review Rate'
    ],
    'Value': [
        average_review_score,
        five_star_rate
    ]
})

review_kpis

,Metric,Value
0,Average Review Score,4.086421
1,5-Star Review Rate,57.776344


###     9.2 Impact of Late Delivery on Reviews

In [41]:
satisfaction_df = (
    delivered_orders[
        [
            'order_id',
            'delivery_days',
            'delivery_delay_days',
            'is_late'
        ]
    ]
    .merge(
        reviews[
            [
                'order_id',
                'review_score'
            ]
        ],
        on='order_id',
        how='inner'
    )
)

In [42]:
review_by_delivery = (
    satisfaction_df
    .groupby('is_late')['review_score']
    .mean()
)

review_by_delivery

is_late
False    4.289999
True     2.271025
Name: review_score, dtype: float64

## 10. Business Recommendations

### 1. Improve Delivery Performance

Approximately **6.77% of delivered orders arrived later than the estimated delivery date**. The business should investigate the causes of late deliveries and improve coordination with logistics and delivery partners. Reducing delivery delays could improve both operational performance and customer satisfaction.

### 2. Focus on High-Value Product Categories

Product categories such as **beleza_saude** and **relogios_presentes** generated substantial revenue. The business should continue to monitor and strategically support high-value categories through inventory management, targeted marketing, and promotional campaigns.

### 3. Improve Customer Retention

The repeat customer rate was approximately **3.12%**, indicating that most customers made only one recorded purchase during the analysis period. The business should consider loyalty programs, personalized promotions, and post-purchase engagement strategies to encourage repeat purchases.

### 4. Improve Customer Satisfaction Through Reliable Delivery

Customers with on-time or early deliveries gave an average review score of approximately **4.29**, while customers with late deliveries gave an average score of approximately **2.27**. This significant difference suggests that improving delivery reliability should be a priority for improving customer satisfaction.

### 5. Address Categories with Delivery Challenges

Some product categories, particularly **moveis_escritorio**, had longer average delivery times. The business should investigate whether product size, shipping complexity, inventory location, or logistics processes contribute to these delays.


## 11. Final Business Summary

The business analysis identified important opportunities across sales performance, product categories, customer behavior, delivery operations, and customer satisfaction.

The business generated approximately **13.59 million in total revenue** from **99,441 orders**. Most orders were successfully delivered, generating approximately **13.22 million in delivered revenue**.

Product performance varied across categories. **beleza_saude** generated the highest revenue, while **cama_mesa_banho** had the highest order volume. This indicates that high sales volume does not always result in the highest revenue.

The customer base was strongly concentrated in **SP**, with Sao Paulo being the city with the highest number of customers. However, the repeat customer rate of approximately **3.12%** indicates a significant opportunity to improve customer retention.

Delivery performance had a strong relationship with customer satisfaction. Approximately **6.77% of delivered orders arrived late**, and customers who experienced late deliveries gave substantially lower review scores than customers whose orders arrived on time or early.

Overall, the business should focus on improving delivery reliability, strengthening customer retention strategies, and continuing to invest in high-performing and high-value product categories.
